In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
import warnings


In [ ]:
df = pd.read_csv('processed_student_data.csv')
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

model = joblib.load('student_performance_model.pkl')

print("Model loaded successfully")
print("Features:", X.shape[1])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape[0])
print("Val:  ", X_val.shape[0])
print("Test: ", X_test.shape[0])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    model, X_train, y_train,
    cv=kf,
    scoring=['neg_mean_absolute_error',
             'neg_root_mean_squared_error',
             'r2'],
    return_train_score=True
)

print("--- 5-Fold Cross Validation Results ---")
print(f"MAE:  {-cv_results['test_neg_mean_absolute_error'].mean():.4f} "
      f"(+/- {cv_results['test_neg_mean_absolute_error'].std():.4f})")
print(f"RMSE: {-cv_results['test_neg_root_mean_squared_error'].mean():.4f} "
      f"(+/- {cv_results['test_neg_root_mean_squared_error'].std():.4f})")
print(f"R²:   {cv_results['test_r2'].mean():.4f} "
      f"(+/- {cv_results['test_r2'].std():.4f})")

In [ ]:
test_preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2   = r2_score(y_test, test_preds)

print("--- Test Set Regression Metrics ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
def score_to_grade(score):
    if score >= 80:   return 'A'
    elif score >= 70: return 'B'
    elif score >= 60: return 'C'
    elif score >= 50: return 'D'
    else:             return 'F'

y_test_grades  = y_test.apply(score_to_grade)
y_pred_grades  = pd.Series(test_preds).apply(score_to_grade)

print("Grade distribution (actual):")
print(y_test_grades.value_counts().sort_index())

In [ ]:
print("Classification Report (Grade Bands)")
print(classification_report(y_test_grades, y_pred_grades))

In [ ]:
from sklearn.preprocessing import LabelBinarizer

lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test_grades)
y_pred_bin = lb.transform(y_pred_grades)

auc = roc_auc_score(y_test_bin, y_pred_bin,
                     multi_class='ovr', average='weighted')

print(f"Weighted AUC-ROC: {auc:.4f}")

In [ ]:
grades = ['A', 'B', 'C', 'D', 'F']
cm = confusion_matrix(y_test_grades, y_pred_grades, labels=grades)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=grades, yticklabels=grades)
plt.title('Confusion Matrix — Grade Bands')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
residuals = y_test.values - test_preds

plt.figure(figsize=(10, 5))
plt.scatter(test_preds, residuals, alpha=0.4, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Exam Score')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot')
plt.tight_layout()
plt.savefig('residual_plot.png', dpi=150)
plt.show()
print("Saved: residual_plot.png")

In [ ]:
#Rebuild test set with original (unscaled) column names
X_test_df = X_test.copy()
X_test_df['Actual'] = y_test.values
X_test_df['Predicted'] = test_preds
X_test_df['Residual'] = X_test_df['Actual'] - X_test_df['Predicted']
X_test_df['AbsError'] = X_test_df['Residual'].abs()

bias_features = [
    'Access_to_Resources_Low',
    'Parental_Involvement_Low',
    'Learning_Disabilities_Yes'
]

print("--- Bias Check: Mean Absolute Error by Subgroup ---\n")
for col in bias_features:
    if col in X_test_df.columns:
        group = X_test_df.groupby(col)['AbsError'].mean()
        print(f"{col}:")
        print(group.to_string())
        print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, col in enumerate(bias_features):
    if col in X_test_df.columns:
        X_test_df.groupby(col)['AbsError'].mean().plot(
            kind='bar', ax=axes[i], color=['steelblue', 'coral']
        )
        axes[i].set_title(f'MAE by {col.replace("_", " ")}')
        axes[i].set_ylabel('Mean Absolute Error')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Bias Check — Model Error Across Subgroups', y=1.02)
plt.tight_layout()
plt.savefig('bias_check.png', dpi=150)
plt.show()
print("Saved: bias_check.png")